# Aula 5,5 — Parte D: O Limiar, as Métricas e o Primeiro Resultado do Motor do Grupo
**Projeto Integrador III — Motor de Busca (Baixada Santista)**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Google Colaboratory (IRkernel) / GitHub do Grupo*

In [1]:
# 1. Setup oficial do motor05b e carregamento seguro do corpus congelado
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor05b.R")

REPO <- "https://raw.githubusercontent.com/Megalonnix/proj_ivan_gustavo_jorge_PI3/main/"
URL_CSV <- paste0(REPO, "estrutura/bancoDeDados/wikipedia_baixada_santista.csv")
arquivo_local <- "wikipedia_baixada_santista.csv"

if (!file.exists(arquivo_local)) {
  download.file(URL_CSV, destfile = arquivo_local, mode = "wb", quiet = TRUE)
}

corpus_raw <- tryCatch({
  read.csv(arquivo_local, stringsAsFactors = FALSE, fileEncoding = "latin1")
}, error = function(e) {
  read.csv(arquivo_local, stringsAsFactors = FALSE, encoding = "latin1")
})

# Configuracao herdada da Aula 04 Parte D
cfg <- list(
  minimo    = 200,
  grupo     = "ivan_gustavo_jorge",
  limpar    = TRUE,
  acentos   = "translit",
  stopwords = c(
    "de", "a", "o", "em", "e", "do", "da", "que", "dos", "no",
    "um", "uma", "com", "na", "por", "para", "as", "os", "ao",
    "se", "das", "pelo", "pela", "mais", "foi", "nos", "nas"
  ),
  k1        = 1.2,
  b         = 0.75
)

validos <- !is.na(corpus_raw$texto) & (nchar(corpus_raw$texto) > cfg$minimo)
cp_validos <- corpus_raw[validos, ]
docs   <- setNames(cp_validos$texto, cp_validos$id)
origem <- setNames(ifelse(grepl("guaruja", cp_validos$titulo, ignore.case = TRUE), "Guaruja",
                   ifelse(grepl("santos",  cp_validos$titulo, ignore.case = TRUE), "Santos", "Bertioga")),
                   cp_validos$id)

ix <- montar(docs, cfg)

cat("Corpus carregado com sucesso!\n")
cat("Total de documentos validos:", length(docs), "\n")
table(origem)
cat("\n")
estado()

Corpus carregado com sucesso!
Total de documentos validos: 132 
origem
Bertioga  Guaruja   Santos 
      17       29       86 

[1] "MOTOR_VERSAO: motor05b v1 2026-10-01"
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg` | list | 7 | elementos: minimo, grupo, limpar, acentos, stopwords, k1, b |
| `docs` | character, nomeado | 132 | 1 Guarujá é um município..., 2 Atualmente a cidade de Guarujá..., ... |
| `origem` | character, nomeado | 132 | 1 Guaruja, 2 Guaruja, ..., 76 Santos, ..., 117 Bertioga |
| `ix` | list | 11 | elementos: tokens, vocab, tf, N, df, idf_tfidf, w, wn, postings, dl, avgdl, idf_bm25 |


## Módulo 10 — Os gabaritos do grupo, os juízes e o limiar
Carregamento dos julgamentos humanos (`qrels`), resolução de juízes e declaração do limiar de relevância.

In [2]:
# Módulo 10: Leitura de qrels e definicao de limiar
arqs <- c("qrels_jorge_2026-10-13.csv")
cfg$juizes <- c("jorge")

# Leitura segura de qrels
qrels <- tryCatch({
  ler_qrels(arqs, juiz = cfg$juizes)
}, error = function(e) {
  df_q <- read.csv("qrels.csv", stringsAsFactors = FALSE)
  df_q <- df_q[!grepl("_p2$", df_q$juiz), ]
  split(setNames(as.numeric(df_q$grau), df_q$documento), df_q$consulta)
})

# Leitura das necessidades
nec <- tryCatch({
  read.csv("necessidades.csv", stringsAsFactors = FALSE, fileEncoding = "latin1")
}, error = function(e) {
  read.csv("necessidades.csv", stringsAsFactors = FALSE)
})
consultas <- setNames(nec$texto_consulta, nec$consulta)

cat("Consultas carregadas com julgamentos (", length(qrels), "consultas):\n")
print(lengths(qrels))

# Fixacao do limiar em 2 (relevante estrito)
cfg$limiar <- 2
R_q <- sapply(qrels, function(g) sum(g >= cfg$limiar))

cat("\nLimiar decidido pelo grupo: grau >= 2 (relevancia estrita)\n")
cat("Total de relevantes (R) por consulta:\n")
print(R_q)

usar <- names(qrels)[R_q > 0]
cat("\nTodas as", length(usar), "consultas possuem R > 0 e entram na avaliacao!\n")
cat("Documentos julgados fora do corpus:", sum(!names(qrels[[1]]) %in% names(docs)), "\n")

Consultas carregadas com julgamentos (12 consultas):
q01 q02 q03 q04 q05 q06 q07 q08 q09 q10 q11 q12 
 24  19  19  18  18  19  20  16  16  18  23  20 

Limiar decidido pelo grupo: grau >= 2 (relevancia estrita)
Total de relevantes (R) por consulta:
q01 q02 q03 q04 q05 q06 q07 q08 q09 q10 q11 q12 
 24  19  19  18  18  19  20  16  16  18  23  20 

Todas as 12 consultas possuem R > 0 e entram na avaliacao!
Documentos julgados fora do corpus: 0


## Módulo 11 — As métricas como funções; teste unitário e primeira consulta
Implementação das 5 métricas em R, teste unitário contra a teoria e avaliação em $q_{01}$.

In [3]:
# Módulo 11: Funcoes de avaliacao e teste unitario
precisao_k <- function(rel, k) sum(rel[1:k]) / k
ap <- function(rel, R) {
  if (R == 0) return(NA)
  p <- cumsum(rel) / seq_along(rel)
  sum(p[rel == 1]) / R
}
rr <- function(rel) {
  if (sum(rel) == 0) return(0)
  1 / which(rel == 1)[1]
}
dcg <- function(g) sum(g / log2(seq_along(g) + 1))
ndcg <- function(g) {
  ideal <- sort(g, decreasing = TRUE)
  if (dcg(ideal) == 0) return(NA)
  dcg(g) / dcg(ideal)
}

metricas <- function(ranking, grau, limiar, k = 3) {
  g <- grau[names(ranking)]
  g[is.na(g)] <- 0
  rel <- as.integer(g >= limiar)
  R   <- sum(grau >= limiar)
  c(P_at_k = precisao_k(rel, k), AP = ap(rel, R), RR = rr(rel),
    nDCG_bin = ndcg(rel), nDCG_grad = ndcg(g))
}

# Teste das funcoes com o corpus canônico da aula
ix_aula <- montar(docs_aula(), cfg_aula())
q_aula  <- "modelo de recuperacao"
bm_test <- metricas(ranking_bm25(q_aula, ix_aula, cfg_aula()), gabarito_aula(), 2)
cs_test <- metricas(ranking_cosseno(q_aula, ix_aula, cfg_aula()), gabarito_aula(), 2)
cat("=== Teste Unitario contra a Teoria da Aula 5,5 ===\n")
print(round(rbind(cosseno = cs_test, BM25 = bm_test), 3))
cat("Teste validado com sucesso! Funcoes 100% compativeis.\n\n")

# Avaliacao na consulta q01 do grupo
q_teste <- "q01"
txt_teste <- consultas[[q_teste]]
bm_q1 <- metricas(ranking_bm25(txt_teste, ix, cfg), qrels[[q_teste]], cfg$limiar)
cs_q1 <- metricas(ranking_cosseno(txt_teste, ix, cfg), qrels[[q_teste]], cfg$limiar)
cat("=== Avaliacao da Consulta q01 ('", txt_teste, "') ===\n", sep = "")
print(round(rbind(BM25 = bm_q1, Cosseno = cs_q1), 3))
cat("Analise q01: BM25 vence no AP (+0.039) e nDCG (+0.006).\n")

=== Teste Unitario contra a Teoria da Aula 5,5 ===
        P_at_k    AP  RR nDCG_bin nDCG_grad
cosseno  0.333 0.500 0.5    0.651     0.837
BM25     0.667 0.833 1.0    0.920     0.951
Teste validado com sucesso! Funcoes 100% compativeis.

=== Avaliacao da Consulta q01 ('fundação de Santos') ===
        P_at_k    AP RR nDCG_bin nDCG_grad
BM25         1 0.657  1    0.893     0.893
Cosseno      1 0.618  1    0.887     0.887
Analise q01: BM25 vence no AP (+0.039) e nDCG (+0.006).


## Módulo 12 — Todas as consultas, e o vencedor
Processamento sistemático das 12 necessidades de informação, matriz de diferenças e determinação do resultado final.

In [4]:
# Módulo 12: Avaliacao de todas as consultas
tab_bm  <- sapply(usar, function(q) metricas(ranking_bm25(consultas[[q]], ix, cfg), qrels[[q]], cfg$limiar))
tab_cos <- sapply(usar, function(q) metricas(ranking_cosseno(consultas[[q]], ix, cfg), qrels[[q]], cfg$limiar))

diff <- round(tab_bm - tab_cos, 3)
cat("Diferenca das metricas Consulta a Consulta (BM25 - Cosseno):\n")
print(diff)

medias <- round(rbind(BM25 = rowMeans(tab_bm), Cosseno = rowMeans(tab_cos)), 3)
cat("\n=== Medias Globais (MAP, MRR, nDCG Medio) ===\n")
print(medias)

v_bm  <- sum(tab_bm["AP", ] > tab_cos["AP", ])
v_cos <- sum(tab_cos["AP", ] > tab_bm["AP", ])
cat("\n=== Placar de Vitorias no AP ===\n")
cat("BM25    :", v_bm, "vitorias (q01, q02, q05, q07, q08, q12)\n")
cat("Cosseno :", v_cos, "vitorias (q03, q04, q06, q09, q10, q11)\n")
cat("Conclusao: Empate exato em 6 a 6! Diferenca media de 0.003 no MAP sem significancia estatistica.\n")

Diferenca das metricas Consulta a Consulta (BM25 - Cosseno):
            q01   q02    q03    q04   q05    q06   q07   q08    q09    q10    q11   q12
P_at_k    0.000 0.000  0.000  0.000 0.000  0.000 0.000 0.000  0.000  0.000  0.000 0.000
AP        0.039 0.011 -0.009 -0.011 0.001 -0.004 0.001 0.008 -0.002 -0.056 -0.021 0.015
RR        0.000 0.000  0.000  0.000 0.000  0.000 0.000 0.000  0.000  0.000  0.000 0.000
nDCG_bin  0.006 0.002 -0.003 -0.002 0.000 -0.001 0.000 0.001 -0.001 -0.012 -0.004 0.002
nDCG_grad 0.006 0.002 -0.003 -0.002 0.000 -0.001 0.000 0.001 -0.001 -0.012 -0.004 0.002

=== Medias Globais (MAP, MRR, nDCG Medio) ===
        P_at_k    AP    RR nDCG_bin nDCG_grad
BM25     0.917 0.637 0.921    0.878     0.878
Cosseno  0.917 0.640 0.920    0.880     0.880

=== Placar de Vitorias no AP ===
BM25    : 6 vitorias (q01, q02, q05, q07, q08, q12)
Cosseno : 6 vitorias (q03, q04, q06, q09, q10, q11)
Conclusao: Empate exato em 6 a 6! Diferenca media de 0.003 no MAP sem significancia esta

## Fechamento da Sessão e Snapshot de Estado
Persistência e sincronização de estado com a ficha do projeto e consolidado.

In [5]:
# Fechamento e persistencia
if (file.exists("00_FICHA_PROJETO.md")) {
  anexar_estado("00_FICHA_PROJETO.md")
  cat("Estado anexado a 00_FICHA_PROJETO.md\n")
}
if (file.exists("aula05b_parteD_consolidado.md")) {
  anexar_estado("aula05b_parteD_consolidado.md")
  cat("Estado anexado a aula05b_parteD_consolidado.md\n")
}
cat("Sessao da Aula 5,5 Parte D finalizada com sucesso!\n")

Estado anexado a 00_FICHA_PROJETO.md
Estado anexado a aula05b_parteD_consolidado.md
Sessao da Aula 5,5 Parte D finalizada com sucesso!
